In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
file_path = "/Volumes/ecommerce_lakehouse/raw/oltp_landing/payments/olist_order_payments_dataset.csv"

column_names = spark.read \
        .format("csv") \
        .option("header","true") \
        .load(file_path) \
        .limit(5)


display(column_names)

In [0]:
payment_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("payment_sequential", IntegerType(), True),
    StructField("payment_type", StringType(), True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value", DoubleType(), True)
])

In [0]:
payments_stream_df = spark.readStream \
                .format("cloudFiles") \
                .option("cloudFiles.format", "csv") \
                .option("header", "true") \
                .schema(payment_schema) \
                .load("/Volumes/ecommerce_lakehouse/raw/oltp_landing/payments/")

In [0]:
bronze_payment_df = payments_stream_df \
        .withColumn("ingestion_timestamp", current_timestamp()) \
        .withColumn("load_date", current_date()) \
        .withColumn("source_file", col("_metadata.file_path"))



In [0]:
query = (
    bronze_payment_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/ecommerce_lakehouse/raw/checkpoints/payments/")
    .trigger(availableNow=True)
    .toTable("ecommerce_lakehouse.bronze.payments_raw")
)

In [0]:
%sql
SELECT *
FROM ecommerce_lakehouse.bronze.payments_raw
LIMIT 10;